###**Abstract**
The age Evolution of Self-Portrait (1800-1940)

*   **Research question**: Did the late 19th century explosion of avant-gardes movements lower the average age at which artists chose to portray themselves?

*   **Hypothesis**: it is initially hypothesized that avant-garde movement, given their countercultural and rebellious youth core, would drastically lower the average age of artists producing self-portraits.

*  **Methodology**: The project combines both a temporal and topical analysis. The research involves the Data Interation of two datasets: *ArtResearch* (the main corpus of self portraits) and *Wikidata* (to integrate information of the artists' biographical data and art movement.)

Project deveolped for the *Information visualization* course, y.a. 2025/2026.



###**Acquire and Parse**: collecting the data
To correctly query the ArtResearch SPARQL Endpoint, an exploring phase of the CIDOC-CRM was necessary, and as explained in the ontology pattern's documentation (*Pharos Data Model*), the art work is modeled as ``` Visual_item ```.

I, then, used a preliminary query to analyze the triples associated with one random self-portait.

```
# PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?property ?value ?valueLabel WHERE {
  <https://data.rkd.nl/images/7675> ?property ?value
  OPTIONAL { ?value rdfs:label ?valueLabel }
}
```
In particular one of the triples resulting was:
```
"http://www.cidoc-crm.org/cidoc-crm/P65_shows_visual_item" , "https://data.rkd.nl/.well-known/genid/dfd37511118b4c84a5c974885baad959" ,
```

In [6]:
"property" , "value" , "valueLabel" ,
"http://www.cidoc-crm.org/cidoc-crm/P1_is_identified_by" , "https://data.rkd.nl/.well-known/genid/e3c88f2e05dd452ebdf5018398de1c97" ,
"http://www.cidoc-crm.org/cidoc-crm/P1_is_identified_by" , "https://data.rkd.nl/.well-known/genid/fbbf1de5d8c34cdcb17f101a0c564289" ,
"http://www.cidoc-crm.org/cidoc-crm/P1_is_identified_by" , "https://data.rkd.nl/images/7675/catalog_url" ,
"http://www.cidoc-crm.org/cidoc-crm/P1_is_identified_by" , "https://data.rkd.nl/.well-known/genid/5713900d6f8d4399aad004a43ebe903f" ,
"http://www.cidoc-crm.org/cidoc-crm/P108i_was_produced_by" , "https://data.rkd.nl/.well-known/genid/14df79ce8dd44ea08be04a005579ed25" ,
"http://www.cidoc-crm.org/cidoc-crm/P2_has_type" , "http://vocab.getty.edu/aat/300033618" ,
"http://www.cidoc-crm.org/cidoc-crm/P2_has_type" , "http://vocab.getty.edu/aat/300133025" ,
"http://www.cidoc-crm.org/cidoc-crm/P2_has_type" , "https://data.rkd.nl/thesaurus/62510" ,
"http://www.cidoc-crm.org/cidoc-crm/P45_consists_of" , "http://vocab.getty.edu/aat/300015050" ,
"http://www.cidoc-crm.org/cidoc-crm/P50_has_current_keeper" , "https://data.rkd.nl/collectors/17040" ,

#this the triple
"http://www.cidoc-crm.org/cidoc-crm/P65_shows_visual_item" , "https://data.rkd.nl/.well-known/genid/dfd37511118b4c84a5c974885baad959" ,

"http://www.cidoc-crm.org/cidoc-crm/P70i_is_documented_in" , "https://artresearch.net/resource/e31/rkd" ,
"http://www.w3.org/1999/02/22-rdf-syntax-ns#type" , "http://www.cidoc-crm.org/cidoc-crm/E22_Human-Made_Object" , "Human-Made Object" ,
"http://www.w3.org/1999/02/22-rdf-syntax-ns#type" , "http://www.cidoc-crm.org/cidoc-crm/E22_Human-Made_Object" , "Objet élaboré par l’humain" ,
"https://artresearch.net/custom/has_image" , "true" ,
"https://artresearch.net/custom/work_preferred_photo" , "https://data.rkd.nl/digital-objects/11118675" ,
"https://artresearch.net/custom/work_thumbnail_url" , "/cache/images/thumbnails/rkd/aHR0cHM6Ly9tZWRpYS5ya2QubmwvaWlpZi85Mjc0MjkwL2Z1bGwvbWF4LzAvZGVmYXVsdC5qcGc.jpg" ,


('https://artresearch.net/custom/work_thumbnail_url',
 '/cache/images/thumbnails/rkd/aHR0cHM6Ly9tZWRpYS5ya2QubmwvaWlpZi85Mjc0MjkwL2Z1bGwvbWF4LzAvZGVmYXVsdC5qcGc.jpg')

This allowed me to find the path for the corresponding Iconclass concept (http://iconclass.org/48B3) of self-portait, which can be summarized in:
```
[Artwork] ──(crm:P65_shows_visual_item)──> [Anonymous Node] ──(crm:P2_has_type)──> Iconclass/48B3.
```

Then, to retrieve the author and date, I identified the ```production``` node which was used in the *Pathos* documentation to connexct both the ```crm:P14_carried_out_by``` and ```crm:P4_has_time-span```.

The final query then becomes:

```
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>

SELECT DISTINCT ?work ?artistURI ?dateBegin WHERE {
  ?work crm:P65_shows_visual_item ?visualItem .
  
  #Linking the visualItem to the type I previously found
  ?visualItem crm:P2_has_type <http://iconclass.org/48B3> .
  
  #Connecting the work to the artist (using production)
  ?work crm:P108i_was_produced_by ?production .
  ?production crm:P14_carried_out_by ?artistURI .
  
  #Choosing a specific time stamp
  ?production crm:P4_has_time-span ?timeStamp .
  ?timeStamp crm:P82a_begin_of_the_begin ?dateBegin .
  
  ?timeStamp crm:P82a_begin_of_the_begin ?dateBegin .
  
  #Adding the specification of the ts
  FILTER(YEAR(?dateBegin) >= 1800)
}

ORDER BY ?dateBegin
LIMIT 2000

```



```
#some of the results
{
  "head": {
    "vars": [
      "work",
      "artistURI",
      "dateBegin"
    ]
  },
  "results": {
    "bindings": [
      {
        "work": {
          "type": "uri",
          "value": "https://artresearch.net/resource/frick/work/991013091589707141"
        },
        "artistURI": {
          "type": "uri",
          "value": "http://vocab.getty.edu/ulan/500002618"
        },
        "dateBegin": {
          "datatype": "http://www.w3.org/2001/XMLSchema#dateTime",
          "type": "literal",
          "value": "1800-01-01T00:00:00Z"
        }
      },
      {
        "work": {
          "type": "uri",
          "value": "https://artresearch.net/resource/frick/work/991001524549707141"
        },
        "artistURI": {
          "type": "uri",
          "value": "https://artresearch.net/resource/frick/actor/faneactive_19th_century"
        },
        "dateBegin": {
          "datatype": "http://www.w3.org/2001/XMLSchema#dateTime",
          "type": "literal",
          "value": "1800-01-01T00:00:00Z"
        }
      },
      {
        "work": {
          "type": "uri",
          "value": "https://artresearch.net/resource/frick/work/991003944339707141"
        },
        "artistURI": {
          "type": "uri",
          "value": "http://www.wikidata.org/entity/Q6220559"
        },
        "dateBegin": {
          "datatype": "http://www.w3.org/2001/XMLSchema#dateTime",
          "type": "literal",
          "value": "1800-01-01T00:00:00Z"
        }
      },
    ]
  }
}
```

To build a comprehensive dataset, I applied the **"Dump and merge"** strategy:

1.   **Dump:** downloaded the result of the ArtResearch query in JSON format and parsed it as a Pandas Dataframe ```df_artresearch```

2. **Merge:** complementary archive query to add missing information like bibliographical and historical details, querying remotely the *Wikidata* SPARQL Endpoint with the ``` SPARQLWrapper``` Python library. The resulting JSON response is, then, parsed into a second Dataframe ```df_wikidata```and merged using Pandas method (```pd.merge```).


In [7]:
import json
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON

#Parsing the result from Artresearch
with open("SelfPortraits.json", "r", encoding="utf-8") as f:
    data_json = json.load(f)

#extracting and normalizing the data in a list of dict
#structure of the json.
# {
#   "head": {
#     "vars": ["work", "artistURI", "dateBegin"]  # Variables of the query
#   },
#   "results": {
#     "bindings": [
#       {
#         "work": {
#           "type": "uri",
#           "value": "https://artresearch.net/resource/frick/work/991013091589707141"
#         },
#         "artistURI": {
#           "type": "uri",
#           "value": "http://vocab.getty.edu/ulan/500002618"
#         },
#         "dateBegin": {
#           "datatype": "http://www.w3.org/2001/XMLSchema#dateTime",
#           "type": "literal",
#           "value": "1800-01-01T00:00:00Z"
#         }
#       },

records = []
for binding in data_json["results"]["bindings"]:
    records.append({
        "work": binding["work"]["value"],
        "artistURI": binding["artistURI"]["value"],
        "dateBegin": binding["dateBegin"]["value"]
    })

#Creating DataFrame with Pandas
df_artresearch = pd.DataFrame(records)
print(f"Successful operation: {len(df_artresearch)} lines uploaded")

#Get the list of artists, I need to separate the Getty and Wikidata values because it's not working
ulan_ids = set()
wikidata_uris = set()

for uri in df_artresearch["artistURI"]:
  uri_str = str(uri).strip()
  if "ulan/" in uri_str:
    #taking just the final number
    ulan_ids.add(f'"{uri_str.split("ulan/")[-1]}"')
  elif "wikidata.org/entity/" in uri_str:
    wikidata_uris.add(f'<{uri_str}>')

ulan_values = " ".join(ulan_ids)
wikidata_values = " ".join(wikidata_uris)

#preparing the query
wikidata_endpoint = "https://query.wikidata.org/bigdata/namespace/wdq/sparql"
query_data = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>


SELECT DISTINCT ?artistURI ?artistLabel ?birthDate ?movementLabel WHERE {
    {
          #Asking for the values we found in Artresearch, in wikidata URIS
          VALUES ?artistURI { """ + wikidata_values + """ } .

          ?artistURI wdt:P569 ?birthDate . #wikidata: birthdate
          ?artistURI rdfs:label ?artistLabel . #adding also the artist Label for a better human-readability
          FILTER (langMatches(lang(?artistLabel), "EN"))

          OPTIONAL{
              ?artistURI wdt:P135 ?movement . #movement
              ?movement rdfs:label ?movementLabel .
              FILTER (langMatches(lang(?movementLabel), "EN"))
          }
    }
    #correspondence with ulan URI
    UNION
    {
          VALUES ?ulan_code { """ + ulan_values + """ } .

          ?artist wdt:P245 ?ulan_code . #union list of artist name
          ?artist wdt:P569 ?birthDate .

          ?artist rdfs:label ?artistLabel .
          FILTER (langMatches(lang(?artistLabel), "EN"))

          #Reconstructing the original Getty URI
          BIND(URI(CONCAT("http://vocab.getty.edu/ulan/", ?ulan_code)) AS ?artistURI) .

          OPTIONAL{
            ?artist wdt:P135 ?movement .
            ?movement rdfs:label ?movementLabel .
            FILTER (langMatches(lang(?movementLabel), "EN"))
          }
      }
    }
"""

#Sending the query
sparql_wd = SPARQLWrapper(wikidata_endpoint)

sparql_wd.setMethod("POST")

sparql_wd.setQuery(query_data)
sparql_wd.setReturnFormat(JSON)

print("Processing the query")
results_wd = sparql_wd.query().convert()

#manipulating the results
wd_records = []
for result in results_wd["results"]["bindings"]:
    wd_records.append({
        "artistURI": result["artistURI"]["value"],
        "artistName": result["artistLabel"]["value"] if "artistLabel" in result else None,
        "birthDate": result["birthDate"]["value"] if "birthDate" in result else None,
        "movement": result["movementLabel"]["value"] if "movementLabel" in result else None
    })
df_wikidata = pd.DataFrame(wd_records)

#Data integration and age calculation using Pandas
df_merged = pd.merge(df_artresearch, df_wikidata, on="artistURI", how="left")

#extracting the age and the work's period to see the age
df_merged["year_work"] = df_merged["dateBegin"].str.extract(r'(\d{4})').astype("Int64")
df_merged["year_birth"] = df_merged["birthDate"].str.extract(r'(\d{4})').astype("Int64")

df_merged["artist_age"] = df_merged["year_work"] - df_merged["year_birth"]

df_final = df_merged[["work", "artistURI","artistName","year_work", "year_birth", "artist_age", "movement"]].drop_duplicates()
df_final.to_csv("SelfPortraits_integrated_dataset.csv", index=False)

print("Process completed!")

Successful operation: 887 lines uploaded
Processing the query
Process completed!


```
https://artresearch.net/resource/frick/work/991004123399707141,http://vocab.getty.edu/ulan/500028037,Jean-Auguste-Dominique Ingres,1800,1780,20,neoclassicism
https://artresearch.net/resource/frick/work/991003455269707141,http://vocab.getty.edu/ulan/500014391,Philipp Otto Runge,1801,1777,24,German Romanticism
https://artresearch.net/resource/frick/work/991002240479707141,http://vocab.getty.edu/ulan/500025767,William Nicholson,1801,1781,20,
https://artresearch.net/resource/frick/work/991001818729707141,http://vocab.getty.edu/ulan/500031518,Francis Legatt Chantrey,1801,1781,20,
https://artresearch.net/resource/frick/work/991001818729707141,http://vocab.getty.edu/ulan/500031518,Francis Leggatt Chantrey,1801,1781,20,
https://artresearch.net/resource/frick/work/991000272309707141,http://vocab.getty.edu/ulan/500003683,Thomas Gimbrede,1801,1781,20,
https://artresearch.net/resource/frick/work/991012190059707141,http://vocab.getty.edu/ulan/500021275,Johann Dominik Bossi,1801,1767,34,
https://artresearch.net/resource/frick/work/991013456359707141,http://vocab.getty.edu/ulan/500027085,Charles Fraser,1802,1782,20,
https://artresearch.net/resource/frick/work/991013137649707141,http://vocab.getty.edu/ulan/500028676,William Allan,1802,1782,20,
https://artresearch.net/resource/frick/work/991004497209707141,http://vocab.getty.edu/ulan/500027085,Charles Fraser,1802,1782,20,
```


###**Filter and Mine**: cleaning the data
In the previous script, after merging the datasets, I already calculated the age of the artists at the time of the work, combining (```year_work ```-```year_birth ```).

The resulting dataset contained some inconstistences:
| Problem | Solution |
| :--- | :--- |
| Some ages were clearly wrong, probably for an incorrect date association | Filtering plausible ages for art production (10-90) |
| The movements are considered as two different instances if they have lower case or upper case |Noramlization with ```.capitalize()``` |
| When there are multiple date possible for a work, the systems duplicates the artwork, necessary to choose one | Removal of duplicated and ```NaN``` |

In [8]:
import pandas as pd

df = pd.read_csv("SelfPortraits_integrated_dataset.csv")

#1.Normalization of movement names
df["movement"] = df["movement"].str.strip().str.capitalize()

#2.Deleting the artist without Date of birth
df_clean = df.dropna(subset=['artist_age']).copy()

#3.Deleting extremes, age range 10 - 90
df_clean = df_clean[(df_clean['artist_age'] >= 10) & (df_clean['artist_age'] <= 90)]

#4.Deleting duplicates, taking the first occurence
df_clean = df_clean.drop_duplicates(subset=['work', 'artistURI'])

#Deleting the float part after the year_birth - age_birth
df_clean['year_birth'] = df_clean['year_birth'].astype(int)
df_clean['artist_age'] = df_clean['artist_age'].astype(int)

df_clean.to_csv('SelfPortraits_cleaned.csv', index=False)
print(f"Process completed. Saved records: {len(df_clean)} lines")

Process completed. Saved records: 683 lines


###**Represent**: data profiling and exploration
Before proceding with the final visualiazion, it is necessary to perform an exploration of the data in order to verify the corpus' representativeness and integrity

**1. Statistical profiling:**

Using the ```.describe()``` method on quantitative variables (```artist_age, year_work```), I checked on the distribution of my corpus.

In [9]:
import pandas as pd

#dataset
df = pd.read_csv("SelfPortraits_cleaned.csv")

macro_stats = df[["year_work", "artist_age"]].describe()

#adding this passage to have integers and not floats
macro_stats_cleaned = macro_stats.apply(lambda col: col.map(lambda x: f"{int(x)}" if x.is_integer() else f"{x:.2f}"))

print("Dataset profiling")
print(macro_stats_cleaned)

Dataset profiling
      year_work artist_age
count       683        683
mean    1860.40      27.17
std       36.43      14.24
min        1800         10
25%        1833         20
50%        1858         20
75%     1885.50      28.50
max        1974         85


The results reveal:
*   **Data quality:** age boundaries (min 10, max 85) confirms the absence of outliers.
*   **Age Distribution:** the median age of 20 shows a strong asymmetry, suggesting the self-portrait is predominantly a youthful phenomenon.
*   **Temporal Distribution:** The median year of production being 1858 splits the century symmetrically, but it nees to be noted the 75% is before 1885, proving what was previously noted, an imabalance towards the first half of the century.

**2. Topical Analysis**:

To check for representativity and ensure the dataset is not skewed by a few famous painters or movements, I analyzed the distribution of  ```artistName```and ```movement``` (the categorical variables).

In [16]:
#Profiling of artists (checking for representativity)
df_artists = df[df['artistName'] != 'Unknown']
top_artists = df_artists["artistName"].value_counts().head(10).reset_index()
top_artists.columns = ["Artist Name", "Number of works"]

print("Representativty check: top 10 artists")
print(top_artists)

#2. Distribution of movements
top_movements = df["movement"].value_counts().head(10).reset_index()
top_movements.columns = ["Artistic Movement", "Number of works"]

print("______________________________________")
print("Represantivity check: top 10 movements")
print(top_movements)

Representativty check: top 10 artists
                    Artist Name  Number of works
0                  Paul Cézanne               15
1  George Peter Alexander Healy               15
2               Eastman Johnson               10
3         William Merritt Chase               10
4              Vincent van Gogh                9
5                   Edgar Degas                9
6                  Thomas Sully                7
7               Chester Harding                7
8                  Paul Gauguin                7
9          William Sidney Mount                6
______________________________________
Represantivity check: top 10 movements
     Artistic Movement  Number of works
0        Impressionism               67
1              Realism               31
2   Post-impressionism               24
3          Romanticism               22
4          Orientalism               17
5        Neoclassicism               15
6        Expressionism               15
7  Hudson river school   

The results reveal:

*   **Artist distribution:** the top artist, Paul Cezanne with 15 works, still covers a minimal fraction of the 683 total records, ensuring authorial representativeness.
*   **Movements distribution:** The top 10 movements cover the majority of the dataset but they are balanced between traditional currents (Neoclassicism, Romanticism) and modern ones (Impressionism, Expressionism, French-Realism). This confirms the dataset is fit to analyze the age variation across the historical art shift of the XIX century.

**3. Temporal Analysis:**
To trace historical evolution of the phenomenon without the flactuation of the singolar years, data was aggregated by decade, specifically dividing the ```year_work``` into 10 year intervalls to compute the median and average.

In [11]:
#Temporal analysis
df_unique_works = df.drop_duplicates(subset=["work"]).copy()

#Extracting decades
df_unique_works["decade"] = (df_unique_works["year_work"] // 10) * 10

#Grouping and calculating the average, mean and count
decade_exploration = df_unique_works.groupby("decade")["artist_age"].agg(["mean", "median", "count"]).reset_index()

#Reducing the range to the core of the dataset
decade_exploration = decade_exploration[(decade_exploration["decade"] <= 1940)]

print("Temporal analysis: average and median by decade")
print(decade_exploration.to_string(index=False))

Temporal analysis: average and median by decade
 decade      mean  median  count
   1800 21.912281    20.0     57
   1810 23.371429    20.0     35
   1820 24.032258    20.0     62
   1830 21.531646    20.0     79
   1840 24.703125    20.0     64
   1850 26.446429    20.0     56
   1860 26.450704    20.0     71
   1870 28.603774    20.0     53
   1880 29.912281    20.0     57
   1890 31.222222    20.0     27
   1900 28.186047    20.0     43
   1910 33.617647    25.0     34
   1920 43.400000    45.0     25
   1930 34.818182    31.0     11
   1940 43.285714    39.0      7


The results reveal:

*   **Corpus asymmetry:** it confirms what was already noted, the dataset is skewed with more data in the first part of the century.
*   **First trend pattern:** the median remains constant for all the XIX century showing a general inclination for younger artist to engage with this type of portrait. **But**, the mean shows the average age grew progressively decade by decade. This means **the original hypothesis has already been disproven**, the new movements and the avant-gardes tend to engage with self-portraits more in the late years.

I need to explore the data through visualization:
1.   **The ordinal values:** a temporal analysis is useful to visualize how the average changes decade by decade.

In [12]:
import plotly.express as px
import pandas as pd

fig_line = px.line(
    decade_exploration,
    x="decade",
    y="mean",
    title="Temporal analysis: average age by decade",
    labels={"decade": "Decade", "mean": "Average Age"}
)

fig_line.update_layout(
    xaxis_title="Decade",
    yaxis_title="Average Age",
    height= 500
)

fig_line.show()

This shows what was already stated in the tabular analysis: the average grows incrementally during the whole century.

2. **Topical analysis:** putting the data in context and seeing, by movement, if the average changes.

In [13]:
#Topical analysis: average age and movements
top_movements = df_clean["movement"].value_counts().head(10).index
df_top_movements = df_clean[df_clean["movement"].isin(top_movements)].copy()

#To have a temporal reference we calculate the mean
chronological_order = df_top_movements.groupby("movement")["year_work"].mean().sort_values().index

#Calculating the average age per movement
df_movement_age = df_top_movements.groupby("movement")["artist_age"].mean().reset_index()
df_movement_age['movement'] = pd.Categorical(df_movement_age['movement'], categories=list(chronological_order), ordered=True)
df_movement_age = df_movement_age.sort_values('movement')

#deleting the legend
df_movement_age['bar_color'] = ['#4a4a4a' if i % 2 == 0 else '#cdcdcd' for i in range(len(df_movement_age))]

fig_bar = px.bar(
    df_movement_age,
    x="movement",
    y="artist_age",
    title="Topical analysis: average artist age per movement",
    category_orders={"movement": list(chronological_order)}
)

bw_colors = ['#4a4a4a' if i % 2 == 0 else '#cdcdcd' for i in range(len(df_movement_age))]
fig_bar.update_traces(marker_color=bw_colors)

#too unclear i'll add an horizonal line at the median
fig_bar.add_hline(
    y=25,
    line_dash="dash",
    line_color="#cb4335",
    line_width=2.5,
    annotation_text="<b>Dataset Median</b>",
    annotation_position="top",
    annotation_font=dict(size=12, color="#cb4335")
)

fig_bar.update_layout(
    xaxis_title_text="Artistic Movements (in chronological order)",
    yaxis_title_text="Average age of artist",
    xaxis_tickangle=45,
    height=500,
    template="plotly_white"
)

The result reveal:
1) The median remains of a young age
2) The two graphs seem coherent in showing a growing trend

This graph shows two problems:
*   This type of visualization is not effective to visualize a pattern
*   The data available is not enough to draw conclusions if I look at the singular movements, as they have different amount of data that would require a pre-process to be fully compared.

Therefore, to show the data effectively and to avoid any statistical distortion, the most effective way to represents this phenomenon is a **scatter plot**: every instance is a dot and it doesn't need any process of normalization which could interfere with the data.

Also, it allows me to show both the temporal aspect (the x-axis) and the topical one (by coloring according to the movement).

In [14]:
import plotly.graph_objects as go
import numpy as np

# 1. Filter the data
top_movements = df_clean["movement"].value_counts().head(10).index
df_top_movements = df_clean[df_clean["movement"].isin(top_movements)].copy()

# 2. Creating the graph with go.Scatter
fig_scatter = go.Figure()

# Creating the dots based on the movement
for movement in top_movements:
    d_points = df_top_movements[df_top_movements["movement"] == movement]
    fig_scatter.add_trace(
        go.Scatter(
            x=d_points["year_work"],
            y=d_points["artist_age"],
            mode="markers",
            name=movement,
            marker=dict(size=6, opacity=0.5, line=dict(width=0)),
            hovertext=d_points["movement"],
            hovertemplate="%{hovertext}<br>Year: %{x}<br>Age: %{y:.0f}<extra></extra>",
        )
    )

# 3. Layout
fig_scatter.update_layout(
    title="<b>Distribution of Artist Age in Self-Portraits</b>",
    template="plotly_white",
    width=1000,
    height=650,
    xaxis=dict(title="Year of the Work", showgrid=False, showline=True, linecolor="#4a4a4a"),
    yaxis=dict(title="Artist Age at Self-Portrait", gridcolor="#f0f0f0", showline=True, linecolor="#4a4a4a"),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.22,
        xanchor="center",
        x=0.5,
        bordercolor="#cdcdcd",
        borderwidth=1
    )
)

fig_scatter.show()

###**Refine**
Then, to strengthen visual clarity, I added the following modifications:
1) **Outlier handing:** I set a threshold for x and y-axes for extreme outliers (both in artist age and the sparsely populated decades), ensuring the core data has the right focus without skewing the visual perception.

2) **Minimize chart junk**: I grouped the movements in two macro-categories to limit visual noise, *The Traditional* and the *Modern* ones.

3) To **minimize ink-ratio** and according to **Tidwell's preattentive variables**, I opted for a functional and natural palette using only three colors (*gray*, *green*, *red*). To visualize scarce data I used opacity with  ```MIN_ROBUST_N = 10 ```, making the decade more transparent when supported by less than 10 items.

4)   **Contextual anchor:** one vertical line was added to mark The Art Shift.  This line acts as a visual breaker, guiding the viewer to the threshold of traditional art.

5)  **Highlighting the trend:** To ensure maximum clarity I added a line connecting the average of artists age for movements and decades combined.

In [27]:
import numpy as np
import plotly.graph_objects as go

# 1. Isolating the top 10 movements
top_movements = df_clean["movement"].value_counts().head(10).index
df_top_movements = df_clean[df_clean["movement"].isin(top_movements)].copy()

# 2. SEMPLIFICATION: dividing them in two subgroups to avoid useless noise
traditional = ["Neoclassicism", "Romanticism", "Realism", "Hudson river school", "Academic art", "Orientalism", "French realism"]
df_top_movements["category"] = df_top_movements["movement"].apply(
    lambda x: "Traditional Movements" if x in traditional else "Modern Movements"
)

color_map = {"Traditional Movements": "#B0B0B0", "Modern Movements": "#228B22"}

# This is fundamental for the correct visualization: calculating the mean AND the count
# of both decade and category combination
df_top_movements["decade"] = (df_top_movements["year_work"] // 10) * 10
ribbon = df_top_movements.groupby(["decade", "category"])["artist_age"].agg(
    mean_age="mean", n="size"
).reset_index()

# 3. OPACITY RULE: below this sample size, the segment is drawn lighter
# to signal a weaker statistical basis for that stretch of the trend
MIN_ROBUST_N = 10
LOW_OPACITY = 0.25
HIGH_OPACITY = 1.0

fig_scatter = go.Figure()

for category in ["Traditional Movements", "Modern Movements"]:
    # 4a. The dots: making them with little opacity to minimalize the final effect
    d_points = df_top_movements[df_top_movements["category"] == category]
    fig_scatter.add_trace(
        go.Scatter(
            x=d_points["year_work"],
            y=d_points["artist_age"],
            mode="markers",
            name=category,
            marker=dict(size=6, color=color_map[category], opacity=0.45, line=dict(width=0)),
            hovertext=d_points["movement"],
            hovertemplate="%{hovertext}<br>Year: %{x}<br>Age: %{y:.0f}<extra></extra>",
        )
    )

    # 4b. Decade-mean line, drawn as separate SEGMENTS so each one can carry
    # its own opacity depending on how many works support it
    d_ribbon = ribbon[ribbon["category"] == category].sort_values("decade").reset_index(drop=True)
    d_ribbon["x_mid"] = d_ribbon["decade"] + 5  # half of the decade

    for i in range(len(d_ribbon) - 1):
        n_avg = (d_ribbon.loc[i, "n"] + d_ribbon.loc[i + 1, "n"]) / 2
        seg_opacity = HIGH_OPACITY if n_avg >= MIN_ROBUST_N else LOW_OPACITY
        fig_scatter.add_trace(
            go.Scatter(
                x=[d_ribbon.loc[i, "x_mid"], d_ribbon.loc[i + 1, "x_mid"]],
                y=[d_ribbon.loc[i, "mean_age"], d_ribbon.loc[i + 1, "mean_age"]],
                mode="lines",
                line=dict(color=color_map[category], width=4),
                opacity=seg_opacity,
                showlegend=False,
                hoverinfo="skip",
            )
        )

    # 4c. Markers on top, each one dimmed individually if n is low
    marker_opacities = [
        HIGH_OPACITY if n >= MIN_ROBUST_N else LOW_OPACITY for n in d_ribbon["n"]
    ]
    fig_scatter.add_trace(
        go.Scatter(
            x=d_ribbon["x_mid"],
            y=d_ribbon["mean_age"],
            mode="markers",
            marker=dict(size=8, color=color_map[category], opacity=marker_opacities),
            showlegend=False,
            hovertext=[f"n = {n} works" for n in d_ribbon["n"]],
            hovertemplate="%{hovertext}<extra></extra>",
        )
    )

fig_scatter.add_vline(
    x=1850,
    line_width=2,
    line_dash="dot",
    line_color="#bd392c",
    opacity=0.7,
    annotation_text="<b>1850: The Art Shift</b>",
    annotation_position="top left",
)

fig_scatter.update_layout(
    title="<b>Self-portraits got older, not younger, after 1850</b>",
    template="plotly_white",
    width=1000,
    height=650,
    xaxis=dict(
        title="Year of the Work",
        range=[1797, 1931],
        showgrid=False,
        showline=True,
        linecolor="#4a4a4a",
    ),
    yaxis=dict(
        title="Artist Age at Self-Portrait",
        range=[10, 70],
        gridcolor="#f0f0f0",
        showline=True,
        linecolor="#4a4a4a",
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.22,
        xanchor="center",
        x=0.5,
        title=dict(text="<b>Artistic Group</b>"),
        bordercolor="#cdcdcd",
        borderwidth=1,
    ),
)

fig_scatter.show()

##**Conclusions**

***An unexpected negative result***

The analysis disproved the original hypothesis. *Instead of decreasing the average age artist produced self-portraits increased constantly over the decades.* While self-portraiture remains a predominantly youthful expression, its demographic boundaries progressively expanded after the 1850 visual breaking point.

What this visualization **cannot do** is to show a definitive conclusion for the post 1900 (seeing the corpus is skewed toward the beginning of the 19th century) however, **what it does** is to successfully observe an historical pivotal point that precedes the new century and the avant-gardes.
